# FinLab 資料每日增量更新 (Incremental Update)

本 Notebook 用於每日定時或手動執行 FinLab 資料更新：
1. **讀取本機 CSV**：自動檢查 `data/` 資料夾內現有資料的最新日期（例如 `2026-10-19`）。
2. **設定 API 範圍**：設定 `data.truncate_start = '2026-10-20'`（最新日期的下一天），僅下載新產生的交易日資料，節省記憶體與傳輸頻寬。
3. **合併與儲存**：若有新資料，自動對齊股票欄位並合併至原本的 CSV 檔案；若尚無新資料，則印出提示並安全跳過。
4. **資料驗證**：檢查更新後的日期範圍、股票檔數與熱門標的收盤價。

In [35]:
import os
import datetime as dt
from pathlib import Path
from contextlib import contextmanager
import requests
import pandas as pd
from IPython.display import display

import finlab
from finlab import data

# 登入 FinLab (讀取 ~/.finlab/credentials.json 快取憑證)
finlab.login()

已登入（使用快取憑證）。


In [36]:
# 強制使用雲端資料（每次都重新下載）
data.force_cloud_download = True

In [39]:
# 定義專案 data 目錄路徑 (相容從專案根目錄或 code/ 資料夾執行)
if Path("data").exists():
    DATA_DIR = Path("data")
elif Path("../data").exists():
    DATA_DIR = Path("../data")
else:
    DATA_DIR = Path("../data")
    DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data 目錄路徑: {DATA_DIR.resolve()}")


@contextmanager
def track_download_size():
    """
    監聽 requests 在呼叫期間產生的實際 HTTP 網路傳輸下載流量 (bytes)。
    自動攔截 response headers 的 Content-Length 或 raw content 大小。
    """
    total_bytes = 0
    original_send = requests.Session.send

    def monitored_send(self, request, **kwargs):
        nonlocal total_bytes
        response = original_send(self, request, **kwargs)
        cl = response.headers.get("Content-Length")
        if cl and cl.isdigit():
            total_bytes += int(cl)
        elif hasattr(response, "content") and response.content:
            total_bytes += len(response.content)
        return response

    requests.Session.send = monitored_send
    try:
        yield lambda: total_bytes
    finally:
        requests.Session.send = original_send


def get_latest_date_from_csv(file_path: Path) -> str | None:
    """
    從現有 CSV 檔案中快速取得最新的日期。
    若檔案不存在或為空，回傳 None。
    """
    if not file_path.exists():
        return None
    
    try:
        # 只載入 date 欄位以大幅節省記憶體與載入時間
        df_dates = pd.read_csv(file_path, usecols=["date"])
        df_dates = df_dates.dropna(subset=["date"])
        if df_dates.empty:
            return None
        latest_date = str(df_dates["date"].iloc[-1]).strip()
        return latest_date
    except Exception as e:
        print(f"讀取 {file_path.name} 日期時發生錯誤: {e}")
        return None


def update_finlab_dataset(dataset_id: str = "price:收盤價", csv_name: str = "price_收盤價.csv") -> pd.DataFrame | None:
    """
    增量更新單一 FinLab 資料集並存入 CSV。
    1. 取得現有 CSV 最新日期
    2. 設定 truncate_start 為下一天
    3. 下載新資料並檢驗 API 實際回傳量與網路傳輸大小
    4. 展示新抓取資料並與舊資料合併存檔
    """
    file_path = DATA_DIR / csv_name
    latest_date = get_latest_date_from_csv(file_path)
    
    # 1. 決定下載起始日期
    if latest_date:
        next_day = pd.to_datetime(latest_date) + dt.timedelta(days=1)
        start_str = next_day.strftime("%Y-%m-%d")
        print(f"\n[{dataset_id}]")
        print(f"-> 本機檔案最新日期: {latest_date}")
        print(f"-> 設定 API truncate_start: {start_str}")
        data.truncate_start = start_str
    else:
        print(f"\n[{dataset_id}]")
        print("-> 本機尚未有歷史資料，將全量下載...")
        data.truncate_start = None
        
    data.truncate_end = None  # 抓到最新一天

    # 2. 呼叫 FinLab API (同時監聽實際網路下載流量)
    try:
        with track_download_size() as get_download_size:
            raw_df = data.get(dataset_id)
        net_bytes = get_download_size()
    finally:
        # 下載完成後重設 truncate 避免影響後續操作
        data.truncate_start = None
        data.truncate_end = None

    # 檢查 API 回傳內容
    if raw_df is None or raw_df.empty:
        print("-> FinLab 未回傳任何資料（可能尚無新交易日）。")
        return None
    
    # 轉為一般 pandas DataFrame，解除 FinlabDataFrame 對 index 屬性的鎖定
    new_df = pd.DataFrame(raw_df)

    # 計算資料在記憶體 (RAM) 中的大小
    mem_bytes = new_df.memory_usage(deep=True).sum()
    mem_str = f"{mem_bytes / 1024:.2f} KB" if mem_bytes < 1024 * 1024 else f"{mem_bytes / (1024 * 1024):.2f} MB"

    # 格式化網路下載流量
    if net_bytes > 0:
        net_str = f"{net_bytes / 1024:.2f} KB" if net_bytes < 1024 * 1024 else f"{net_bytes / (1024 * 1024):.2f} MB"
        traffic_info = f"{net_str} ({net_bytes:,} bytes)"
    else:
        traffic_info = "0 KB (自本機快取讀取，未消耗額外網路流量)"

    # 【API 下載與大小檢驗】
    start_str_check = str(new_df.index[0])[:10]
    end_str_check = str(new_df.index[-1])[:10]
    print(f"-> [API 回傳檢驗] 原始下載形狀: {new_df.shape} (天數: {new_df.shape[0]}, 標的數: {new_df.shape[1]})")
    print(f"-> [API 回傳檢驗] 原始下載起訖: {start_str_check} ~ {end_str_check}")
    print(f"-> [資料大小檢驗] 實際網路下載流量: {traffic_info}")
    print(f"-> [資料大小檢驗] 記憶體 (RAM) 佔用: {mem_str}")
    if latest_date:
        if len(new_df) > 30:
            print(f"   [注意] 原始回傳天數高達 {len(new_df)} 天，可能表示 truncate_start 未完全限制到 API 下載量！")
        else:
            print(f"   [成功] truncate_start 生效！API 僅下載了 {len(new_df)} 天的增量資料，大幅節省網路頻寬。")
    
    # 若有舊資料，篩選出真正大於 latest_date 的新天數
    if latest_date:
        latest_dt = pd.to_datetime(latest_date)
        strictly_new = new_df.loc[pd.to_datetime(new_df.index) > latest_dt]
        if strictly_new.empty:
            print(f"-> 已是最新狀態，FinLab 無大於 {latest_date} 的新資料，無需更新。")
            return None
        new_df = strictly_new

    # 3. 直接展示新抓取到的資料 (新天數與代表性標的價格)
    preview_cols = [c for c in ["0050", "0056", "2330", "2317"] if c in new_df.columns]
    print(f"\n-> 【本次新抓到的資料預覽】(共 {len(new_df)} 天):")
    if preview_cols:
        display(new_df[preview_cols])
    else:
        display(new_df.iloc[:, :5])

    # 4. 合併現有舊資料
    if file_path.exists():
        print("\n-> 讀取本機舊資料進行合併...")
        old_df = pd.read_csv(file_path, index_col="date")
        
        # 統一轉成 %Y-%m-%d 字串格式以利合併與排序 (相容舊 CSV 中的斜線格式)
        old_df.index = pd.to_datetime(old_df.index).strftime("%Y-%m-%d")
        new_df.index = pd.to_datetime(new_df.index).strftime("%Y-%m-%d")
        
        # 垂直合併，自動 outer join 處理新舊標的代號
        combined_df = pd.concat([old_df, new_df], axis=0)
        
        # 移除重複日期 (以最新下載為主) 並按日期排序
        combined_df = combined_df[~combined_df.index.duplicated(keep="last")].sort_index()
    else:
        new_df.index = pd.to_datetime(new_df.index).strftime("%Y-%m-%d")
        combined_df = new_df.sort_index()

    # 5. 存回本機 CSV
    print(f"-> 寫入檔案: {file_path.name} (更新後總形狀: {combined_df.shape})")
    combined_df.to_csv(file_path, index_label="date")
    print("-> 更新完成！")
    
    return combined_df

Data 目錄路徑: H:\我的雲端硬碟\DataScience\2026玉山人工智慧賽\data


In [40]:
# 執行收盤價增量更新
close_df = update_finlab_dataset(
    dataset_id="price:收盤價",
    csv_name="price_收盤價.csv"
)


[price:收盤價]
-> 本機檔案最新日期: 2007/4/23
-> 設定 API truncate_start: 2007-04-24
Daily usage: 104.7 / 5000 MB - price:收盤價
-> [API 回傳檢驗] 原始下載形狀: (4790, 2786) (天數: 4790, 標的數: 2786)
-> [API 回傳檢驗] 原始下載起訖: 2007-04-24 ~ 2026-10-19
-> [資料大小檢驗] 實際網路下載流量: 17.22 MB (18,058,540 bytes)
-> [資料大小檢驗] 記憶體 (RAM) 佔用: 101.85 MB
   [注意] 原始回傳天數高達 4790 天，可能表示 truncate_start 未完全限制到 API 下載量！

-> 【本次新抓到的資料預覽】(共 4790 天):


symbol        0050   0056    2330   2317
date                                    
2007-04-24   58.10    NaN    69.8  221.0
2007-04-25   57.60    NaN    69.3  218.0
2007-04-26   57.70    NaN    69.9  221.0
2007-04-27   57.50    NaN    69.0  218.5
2007-04-30   56.90    NaN    68.7  221.5
...            ...    ...     ...    ...
2026-10-13  112.40  58.55  2490.0  240.0
2026-10-14  113.00  59.60  2490.0  243.0
2026-10-15  108.90  58.25  2440.0  239.5
2026-10-16  109.00  58.75  2420.0  244.0
2026-10-19  108.15  58.00  2420.0  239.5

[4790 rows x 4 columns]


-> 讀取本機舊資料進行合併...
-> 寫入檔案: price_收盤價.csv (更新後總形狀: (4791, 2786))
-> 更新完成！


In [34]:
# 讀取並檢視更新後的檔案資訊
target_csv = DATA_DIR / "price_收盤價.csv"
if target_csv.exists():
    # 載入最後 10 天資料檢視
    df_check = pd.read_csv(target_csv, index_col="date")
    print(f"\n檔案路徑: {target_csv.name}")
    print(f"總交易日數 (列數): {df_check.shape[0]}")
    print(f"總標的數 (欄數): {df_check.shape[1]}")
    print(f"歷史起訖範圍: {df_check.index[0]} ~ {df_check.index[-1]}")
    
    # 檢視代表性標的 (0050, 0056, 2330, 2317) 最近 5 日收盤價
    benchmark_codes = [c for c in ["0050", "0056", "2330", "2317"] if c in df_check.columns]
    print("\n--- 代表性標的最近 5 日收盤價 ---")
    display(df_check[benchmark_codes].tail())



檔案路徑: price_收盤價.csv
總交易日數 (列數): 4791
總標的數 (欄數): 2786
歷史起訖範圍: 2007-04-23 ~ 2026-10-19

--- 代表性標的最近 5 日收盤價 ---


              0050   0056    2330   2317
date                                    
2026-10-13  112.40  58.55  2490.0  240.0
2026-10-14  113.00  59.60  2490.0  243.0
2026-10-15  108.90  58.25  2440.0  239.5
2026-10-16  109.00  58.75  2420.0  244.0
2026-10-19  108.15  58.00  2420.0  239.5